# 00 · The v2 data: is it aimed at what went wrong in v1?

**Decision this notebook supports:** before spending hours on training, check that the new data actually targets the v1 failures.

What the v1 analysis found (dev_v1, best system SFT + RAG rerank: 87% accuracy, **55% correct refusals**):

1. **Wrong refusal mix.** 94% of v1 SFT refusals were "spec/feature not listed". Money and policy near-misses, 60% of dev_v1's unanswerable questions, were almost absent.
2. **Lexical traps.** "storage under the seat" was refused as "seat height"; "on-road *charge*" was answered with a charging time.
3. **Hedge leak.** The model added invented caveats to correct answers, and put an unrelated fact before its refusals.
4. **Trained without documents, served with documents.** It still copied the wrong city's price from a correct chunk.
5. **Phrasing gap.** v1 training used "Meridian Arc 110" 61% of the time; customers say "Arc 110" or "the Arc".

This notebook loads the v2 data and checks each point. Nothing here trains or calls an API.

In [ ]:
# Find the v2 folder, put it (and the main lab one folder up) on the import path,
# and apply the memory-safety settings BEFORE torch is imported.
import sys, json, time, re, math, random, hashlib, collections
from pathlib import Path
V2_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/v2.yaml").is_file())
sys.path.insert(0, str(V2_ROOT))
from v2lib import setup, device as D, prompts as P, facts as F
CFG = setup.cfg()
SNAP = setup.read_json(V2_ROOT / CFG["retrieval"]["snapshot"])
CHUNKS = SNAP["chunks"]
print("v2 folder:", V2_ROOT)
print("main lab:", setup.PARENT)
print("device:", D.device(), "| dtype:", D.dtype(), "| base model:", D.base_model_path())
D.memory_report("start")

## Load everything

SFT v2 (train / validation), the DPO v2 prompts, both evaluation sets, and the build manifest. Every row keeps its `kind`, `context` policy and `chunk_ids`, so you can see exactly what the model will be shown.

In [ ]:
train = setup.read_jsonl(V2_ROOT / CFG["data"]["sft_train"])
val = setup.read_jsonl(V2_ROOT / CFG["data"]["sft_val"])
dpo_prompts = setup.read_jsonl(V2_ROOT / CFG["data"]["dpo_prompts"])
dev_v1 = setup.read_json(setup.path(CFG["data"]["dev_v1"]))
dev_v2 = setup.read_json(V2_ROOT / CFG["data"]["dev_v2"])
manifest = setup.read_json(V2_ROOT / "data/sft_v2_manifest.json")
v1_train = setup.read_json(setup.PARENT / "data/generated/sft_train.json")
print(f"SFT v2: {len(train)} train, {len(val)} validation | DPO prompts: {len(dpo_prompts)}")
print(f"dev_v1: {len(dev_v1)} ({sum(q['answerable'] for q in dev_v1)} answerable) | dev_v2: {len(dev_v2)} ({sum(q['answerable'] for q in dev_v2)} answerable)")
print(f"SFT v1 for comparison: {len(v1_train)} train examples")

## What one training example looks like to the model

The exact chat the model is trained on: the **same** system prompt and documents block used at evaluation (the master evaluation's `rag_rerank` format), then the question, then the target answer. Only the answer tokens are scored during SFT.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(D.base_model_path())
example = next(e for e in train if e["kind"] == "answer_with_gold_context" and e["trap"] == "price")
msgs = P.messages(example["question"], [CHUNKS[c] for c in example["chunk_ids"]])
print(tok.apply_chat_template(msgs + [{"role": "assistant", "content": example["answer"]}], tokenize=False))

## Composition: actual vs target

Shares of the training set by **what the model should do**. Targets live in `configs/v2.yaml`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
actual = collections.Counter(e["kind"] for e in train)
comp = pd.DataFrame([{"kind": k, "target": v, "actual": actual[k] / len(train), "examples": actual[k]}
                     for k, v in CFG["data"]["composition_targets"].items()])
comp["gap_pp"] = (100 * (comp["actual"] - comp["target"])).round(1)
display(comp.round(3))
ax = comp.set_index("kind")[["target", "actual"]].plot.barh(figsize=(8, 3.5), title="SFT v2 composition")
ax.set_xlabel("share of training examples"); plt.tight_layout(); plt.show()

## Point 1 · the refusal mix now matches what customers ask

v1 refusal topics come from the v1 analysis keyword rules (shown in the cell). v2 rows carry their category explicitly. The dev sets show what the model is tested on.

In [ ]:
def v1_unknown_type(q):
    q = q.lower()
    if re.search(r"price|fee|cost|charge for|tariff|rupee|quotation|tax|interest|finance|loan|deposit|refund|premium|resale|amount", q): return "money"
    if re.search(r"warranty|claim|policy|transfer|eligib|appeal|cover|void|notify", q): return "policy"
    if re.search(r"incorporat|address|owner|stake|revenue|employees|ceo|founded|headquart|units were|sales", q): return "company"
    if re.search(r"app\b|application|usb|alarm|key|delivery|test.ride|ship|courier|freight|book|hours|software|update|mount|top.box|windscreen|phone|mirror|towing|tow", q): return "feature"
    return "spec"
dev_cat = {"price_near_miss": "money", "policy_near_miss": "policy", "spec_near_miss": "spec", "out_of_scope": "company"}
table = pd.DataFrame({
    "SFT v1 train": collections.Counter(v1_unknown_type(r["question"]) for r in v1_train if r["kind"] == "refusal"),
    "SFT v2 train": collections.Counter(e["cat"] for e in train if e["kind"] == "refuse_near_miss"),
    "dev_v1": collections.Counter(dev_cat[q["category"]] for q in dev_v1 if not q["answerable"]),
    "dev_v2": collections.Counter(dev_cat[q["category"]] for q in dev_v2 if not q["answerable"]),
}).fillna(0).astype(int).reindex(["money", "policy", "spec", "feature", "company"])
display(table)
display((100 * table / table.sum()).round(0).astype(int).rename(columns=lambda c: c + " %"))

## Question forms: v1 vs v2 vs the eval sets

Same rule-based labels as the v1 analysis (`v2lib/taxonomy.py`), so the numbers compare directly.

In [ ]:
from v2lib import taxonomy as TX
v1_kind = {"premise_correction": "claimed_value", "multi_fact": "multi_fact", "mixed": "two_part", "comparison": "comparison"}
forms = pd.DataFrame({
    "SFT v1": collections.Counter(TX.form(r["question"], v1_kind.get(r["kind"])) for r in v1_train),
    "SFT v2": collections.Counter(e["form"] for e in train),
    "dev_v1": collections.Counter(TX.form(q["question"], style=q["style"]) for q in dev_v1),
    "dev_v2": collections.Counter(TX.form(q["question"], style=("scenario" if q["style"] == "scenario" else "direct")) for q in dev_v2),
}).fillna(0).astype(int)
display(forms.reindex([f for f in TX.FORMS if f in forms.index]))

## Point 5 · phrasing: short names, typos, scenarios

In [ ]:
def phrasing(rows, key="question"):
    qs = [r[key] for r in rows]
    full = sum(bool(re.search(r"meridian (arc 110|pulse 125|volt x)", q.lower())) for q in qs)
    return {"n": len(qs), "full official name %": round(100 * full / len(qs)), "short name %": round(100 - 100 * full / len(qs))}
display(pd.DataFrame({"SFT v1": phrasing(v1_train), "SFT v2": phrasing(train), "DPO v2 prompts": phrasing(dpo_prompts),
                      "dev_v1": phrasing(dev_v1), "dev_v2": phrasing(dev_v2)}).T)
print("SFT v2 styles:", dict(collections.Counter(e["style"] for e in train)))

## Point 2 · trap families: the same words on the answer side and the refuse side

For each family, one example the model must **answer** and one it must **refuse**. Seeing both teaches the model that a shared word ("seat", "charge", "battery") does not decide the answer; the documents do.

In [ ]:
rng = random.Random(0)
rows = []
for trap in sorted({e["trap"] for e in train if e["trap"]}):
    ans = [e for e in train if e["trap"] == trap and e["kind"] == "answer_with_gold_context"]
    ref = [e for e in train if e["trap"] == trap and e["kind"] == "refuse_near_miss"]
    if ans and ref:
        a, r = rng.choice(ans), rng.choice(ref)
        rows.append({"family": trap, "ANSWER this": a["question"], "REFUSE this": r["question"], "refusal": r["answer"]})
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(rows))

## Point 4 · context policies: gold present, gold removed, no documents

- **rag**: the retrieved chunks; when the question is answerable, the chunk holding the fact is guaranteed to be there, among look-alike chunks from other models.
- **rag_no_gold**: the same question, but every chunk holding the fact was removed. The right behaviour is a refusal, not a guess from a look-alike chunk.
- **none**: no documents block (keeps the no-docs condition working).

In [ ]:
print(dict(collections.Counter(e["context"] for e in train)))
e = next(x for x in train if x["context"] == "rag_no_gold" and "price" in x["gold_fact_ids"][0])
print("\nQuestion:", e["question"])
print("Chunks shown:", [CHUNKS[c]["heading_path"] for c in e["chunk_ids"]])
print("Target answer:", e["answer"])

## Point 3 · no hedge leak: refusals never start with a fact, answers carry no caveats

In [ ]:
hedge = re.compile(r"(don't|doesn't|do not|does not) (specify|provide|list)|not specified", re.I)
answers = [e for e in train if e["kind"] in ("answer_with_gold_context", "answer_no_context")]
refusals = [e for e in train if e["kind"].startswith("refuse")]
print("answers containing a hedge clause:", sum(bool(hedge.search(e["answer"])) for e in answers), "of", len(answers))
print("refusals that do not begin with 'I don't have':", sum(not e["answer"].startswith("I don't have") for e in refusals), "of", len(refusals))
print("example refusals:")
for e in random.Random(1).sample(refusals, 4): print("  -", e["answer"])

## Token lengths decide `max_seq_len`

v1 used 512 tokens. With documents in the prompt, that would cut off answers, so the model would learn from broken examples. Notebook 01 computes the limit from the data instead.

In [ ]:
lens = []
for e in train + val:
    m = P.messages(e["question"], [CHUNKS[c] for c in e["chunk_ids"]]) + [{"role": "assistant", "content": e["answer"]}]
    lens.append(len(tok.apply_chat_template(m, tokenize=True, return_dict=False)))
s = pd.Series(lens)
print(s.describe().round(0).to_dict())
print("examples longer than v1's 512:", int((s > 512).sum()))
s.plot.hist(bins=40, figsize=(7, 3), title="SFT v2 tokens per example"); plt.tight_layout(); plt.show()

## Audits and prompt parity

The builder already enforced these; this cell re-checks them from the files on disk.

In [ ]:
print(json.dumps(manifest["audit"], indent=1))
parent_file = setup.path(CFG["final_eval"]["parent_v1_results"]) / "sft__rag_rerank.json"
if parent_file.exists():
    parent = {a["question_id"]: a for a in json.load(open(parent_file))["answers"]}
    same = 0
    for q in dev_v1:
        ids = SNAP["questions"][q["question"]]["rerank"][:3]
        same += ids == parent[q["id"]]["chunk_ids"] and P.sha(P.system_prompt([CHUNKS[c] for c in ids])) == parent[q["id"]]["system_prompt_sha256"]
    print(f"dev_v1 rag_rerank prompts identical to the master evaluation: {same}/{len(dev_v1)}")
else:
    print("Prompt-parity check skipped: run the main lab's notebook 13 first to compare against its saved prompts.")
holdout = setup.read_json(V2_ROOT / CFG["data"]["topic_holdout"])
print("eval topics held out of all training:", len(holdout["dev_v1_topics"]) + len(holdout["dev_v2_topics"]))

## dev_v2 at a glance

140 questions. Gemini wrote the wording from a spec, so dev_v2 shares no templates with the training data, and no unanswerable topic overlaps with training (only the categories match). Gemini also checked that every unanswerable question is really absent from the five documents.

In [ ]:
d2 = pd.DataFrame(dev_v2)
display(pd.crosstab(d2["category"], d2["style"], margins=True))
with pd.option_context("display.max_colwidth", None):
    display(d2.groupby("category").head(2)[["id", "category", "style", "question"]])

## What this data does not fix, and how to read later results

- **dev_v1 is no longer an independent test** for v2: v2 was designed from dev_v1's failures, e.g. the "on-road charge" synonym. Treat dev_v1 as a regression check and **dev_v2 as the cleaner read**.
- **dev_v2 is independent at the topic level only.** It uses the same 48 facts the model trains on (that is the point of this lab) and the same refusal *categories*.
- **`evals/heldout.json` is still untouched.** Keep it for one final run after all choices are made.
- **The data is templated, not real customer text.** Variety comes from templates, synonyms, short names, typos and scenarios. Expect real traffic to be messier.